In [13]:
import faultier
import serial
import time

ft = faultier.Faultier()
ser = serial.Serial(ft.get_serial_path(), baudrate=115200)
ser.timeout = 0.05


In [3]:

ser.reset_input_buffer()

ft.configure_glitcher(
    power_cycle_output = faultier.OUT_MUX0,
    power_cycle_length = 300000
)

ft.power_cycle()
d = ser.readline()
print(d)

b'RSTOKAY\x00\x02\x00 xV4\x12\x00\x00\x00\x008\x14\x00 \x01\x00\x00 4S\x00\x00\xefN\x00\x008\x14\x00 \x00\x00\x00\x00\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00/\x00\x00 '


In [14]:
import struct
ft.configure_glitcher(
    power_cycle_output = faultier.OUT_MUX0,
    power_cycle_length = 300000,
    trigger_source = faultier.TRIGGER_IN_EXT0,
    trigger_type = faultier.TRIGGER_PULSE_POSITIVE,
    glitch_output= faultier.OUT_CROWBAR
)

while True:
    # print("Restart")
    for delay in range(56, 60):
        for pulse in range(3, 13):
            # Flush potentially left-over data in serial input buffer
            ser.reset_input_buffer()

            # Glitch with our delay and pulse
            ft.glitch(delay, pulse)

            length = 3 + 4 + (14 * 4) # RST + OKAY + 14 * 4 registers
            # Read in 11 bytes of data
            data = ser.read(length) # RST + OKAY + 14 * 4 registers

            if not data.startswith(b"RSTOKAY"):
                # Ignore failed runs
                continue

            if(len(data) != length):
                continue
            registers = struct.unpack("<" + (14 * "I"), data[7:])

            # print(hex(registers[1]))
            for i in range(0, 14):
                if i == 1:
                    continue
                if registers[i] == 0x12345678:
                    print(f"{delay} {pulse} Register move to r{i}")

            # # Ignore regular execution
            # if(data == b'RSTBRANOKAY'):
            #     ser.read(4) # Read XXXX and ignore it
            #     continue

            # # Just reset
            # if(data == b'RST'):
            #     continue

            # if(data == b'RSTOKAYXXXX'):
            #     print("SKIP {data} {delay} {pulse}")


KeyboardInterrupt: 